In [1]:
"""
HDFS v1 Anomaly Detection - Multi-Classifier Pipeline
======================================================
Based on:
  - Liang et al. (2007) - Failure Prediction in IBM BlueGene/L Event Logs  [SVM]
  - Bodik et al. (2010) - Fingerprinting the Datacenter                    [Logistic Regression / HiLighter]
  - Chen et al. (2004)  - Failure Diagnosis Using Decision Trees            [Decision Tree]

Evaluation: 1 outer cross-validation run × 5-fold Stratified K-Fold
Dataset   : HDFS_v1  (Event_occurrence_matrix + anomaly_label)

Usage
-----
Place the following files in the SAME directory as this script (or adjust
DATA_DIR below):
    anomaly_label.xlsx
    Event_occurrence_matrix.xlsx
    HDFS.npz                   ← used as fast-path if present

Run:
    python hdfs_v1_classification.py
"""

import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score,
    f1_score, classification_report
)

warnings.filterwarnings("ignore")

# ── USER CONFIG ────────────────────────────────────────────────────────────────
DATA_DIR   = "."          # folder that holds the dataset files
N_SPLITS   = 5           # stratified k-fold splits
RANDOM_STATE = 42
# ──────────────────────────────────────────────────────────────────────────────

LABEL_MAP = {"Normal": 0, "Anomaly": 1}   # adjust if your labels differ


# ═══════════════════════════════════════════════════════════════════════════════
# 1. DATA LOADING
# ═══════════════════════════════════════════════════════════════════════════════

def load_dataset(data_dir: str):
    """
    Priority order:
      1. HDFS.npz  (pre-built X, y arrays → fastest)
      2. Event_occurrence_matrix.xlsx + anomaly_label.xlsx  (raw Excel)

    Returns
    -------
    X : np.ndarray  shape (n_samples, n_features)
    y : np.ndarray  shape (n_samples,)  – 0=Normal, 1=Anomaly
    feature_names : list[str]
    """
    npz_path = os.path.join(data_dir, "HDFS.npz")
    eom_path = os.path.join(data_dir, "Event_occurrence_matrix.csv")
    lbl_path = os.path.join(data_dir, "anomaly_label.csv")

    # ── Fast-path: NPZ ────────────────────────────────────────────────────────
    if os.path.exists(npz_path):
        print(f"[INFO] Loading from {npz_path} …")
        npz = np.load(npz_path, allow_pickle=True)
        keys = list(npz.keys())
        print(f"       NPZ keys: {keys}")

        # Common key conventions seen in HDFS benchmarks
        x_key = next((k for k in keys if k.lower() in ("x", "data", "features")), None)
        y_key = next((k for k in keys if k.lower() in ("y", "label", "labels")),  None)

        if x_key and y_key:
            X = npz[x_key].astype(np.float32)
            y_raw = npz[y_key]
            # handle string labels
            if y_raw.dtype.kind in ("U", "S", "O"):
                y = np.array([LABEL_MAP.get(str(v), int(v)) for v in y_raw])
            else:
                y = y_raw.astype(int)
            feature_names = [f"E{i+1}" for i in range(X.shape[1])]
            print(f"       X shape: {X.shape}, y shape: {y.shape}")
            print(f"       Class distribution – Normal: {(y==0).sum()}, Anomaly: {(y==1).sum()}")
            return X, y, feature_names

        print("[WARN] NPZ found but expected keys missing; falling back to Excel.")

    # ── Excel path ────────────────────────────────────────────────────────────
    if not (os.path.exists(eom_path) and os.path.exists(lbl_path)):
        sys.exit(
            f"\n[ERROR] Dataset files not found.\n"
            f"  Expected one of:\n"
            f"    {npz_path}\n"
            f"  OR both of:\n"
            f"    {eom_path}\n"
            f"    {lbl_path}\n"
            f"  Please copy the HDFS_v1 files into: {os.path.abspath(data_dir)}\n"
        )


    print(f"[INFO] Loading from CSV files …")
    eom_df = pd.read_csv(eom_path, index_col=0) 
    lbl_df = pd.read_csv(lbl_path)              

    # Normalize column names
    lbl_df.columns = lbl_df.columns.str.strip()
    block_col  = lbl_df.columns[0]
    label_col  = lbl_df.columns[1]
    lbl_df = lbl_df.set_index(block_col)

    # Align on common block IDs
    common = eom_df.index.intersection(lbl_df.index)
    if len(common) == 0:
        sys.exit("[ERROR] No common block IDs between feature matrix and label file.")

    X_df = eom_df.loc[common]
    y_raw = lbl_df.loc[common, label_col].astype(str).str.strip()
    y = y_raw.map(LABEL_MAP)

    # 1. CLEAN THE DATA
    # Remove non-numeric columns (like "Success" strings)
    X_df = X_df.select_dtypes(include=[np.number])
    
    # --- FIX STARTS HERE ---
    # 2. FILL MISSING VALUES WITH 0
    # This handles the "Input X contains NaN" error
    if X_df.isnull().values.any():
        print(f"[INFO] Missing values detected. Filling NaNs with 0.")
        X_df = X_df.fillna(0)
    # --- FIX ENDS HERE ---

    if X_df.empty:
        sys.exit("[ERROR] No numeric columns found in the feature matrix.")

    # 3. CONVERT TO NUMPY AND RETURN
    X = X_df.values.astype(np.float32)
    y = y.values.astype(int)
    feature_names = list(X_df.columns)

    print(f"       X shape: {X.shape}, y shape: {y.shape}")
    print(f"       Class distribution – Normal: {(y==0).sum()}, Anomaly: {(y==1).sum()}")
    
    return X, y, feature_names  # Don't forget to return!
# ═══════════════════════════════════════════════════════════════════════════════
# 2. CLASSIFIERS  (inspired by the three papers)
# ═══════════════════════════════════════════════════════════════════════════════

def build_classifiers():
    """
    Returns dict of {name: sklearn Pipeline}.

    SVM          → Liang et al. 2007 (LIBSVM / RBF kernel)
    Logistic Reg → Bodik et al. 2010 / HiLighter  (L1-regularised logistic regression)
    Decision Tree→ Chen et al. 2004  (C4.5-style, entropy criterion)
    """
    clfs = {
        "SVM (RBF)": Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    SVC(kernel="rbf", C=1.0, gamma="scale",
                           class_weight="balanced",
                           random_state=RANDOM_STATE,
                           probability=True))
        ]),
        "Logistic Regression (L1)": Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    LogisticRegression(penalty="l1", solver="saga",
                                          C=1.0,
                                          class_weight="balanced",
                                          max_iter=1000,tol=0.01,
                                          random_state=RANDOM_STATE))
        ]),
        "Decision Tree (C4.5-style)": Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    DecisionTreeClassifier(criterion="entropy",
                                              min_samples_leaf=5,
                                              class_weight="balanced",
                                              random_state=RANDOM_STATE))
        ]),
    }
    return clfs


# ═══════════════════════════════════════════════════════════════════════════════
# 3. EVALUATION HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

DIVIDER = "─" * 72

def print_fold_results(fold_idx, clf_name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    p  = precision_score(y_true, y_pred, zero_division=0)
    r  = recall_score   (y_true, y_pred, zero_division=0)
    f1 = f1_score       (y_true, y_pred, zero_division=0)
    print(f"  Fold {fold_idx+1:>2} │ Precision={p:.4f}  Recall={r:.4f}  F1={f1:.4f}")
    return cm, p, r, f1


def plot_normalized_cm(cm_avg, clf_name, axes, idx):
    """Plot normalised confusion matrix (averaged over folds)."""
    cm_norm = cm_avg / cm_avg.sum(axis=1, keepdims=True)
    sns.heatmap(
        cm_norm, annot=True, fmt=".3f", cmap="Blues",
        xticklabels=["Normal", "Anomaly"],
        yticklabels=["Normal", "Anomaly"],
        ax=axes[idx], vmin=0, vmax=1
    )
    axes[idx].set_title(clf_name, fontsize=10, fontweight="bold")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("Actual")


# ═══════════════════════════════════════════════════════════════════════════════
# 4. MAIN PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

def run_pipeline(X, y, classifiers):
    """
    1-cross  ×  5-fold Stratified K-Fold for each classifier.
    Returns dict of per-classifier aggregated results.
    """
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    results = {}

    for clf_name, pipeline in classifiers.items():
        print(f"\n{'═'*72}")
        print(f"  CLASSIFIER: {clf_name}")
        print(DIVIDER)

        fold_cms, fold_p, fold_r, fold_f1 = [], [], [], []

        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)

            cm, p, r, f1 = print_fold_results(fold_idx, clf_name, y_test, y_pred)
            fold_cms.append(cm)
            fold_p.append(p)
            fold_r.append(r)
            fold_f1.append(f1)

        # ── Aggregate ────────────────────────────────────────────────────────
        cm_sum   = np.sum(fold_cms, axis=0)
        avg_p    = np.mean(fold_p)
        avg_r    = np.mean(fold_r)
        avg_f1   = np.mean(fold_f1)
        std_p    = np.std(fold_p)
        std_r    = np.std(fold_r)
        std_f1   = np.std(fold_f1)

        print(DIVIDER)
        print(f"  AVERAGE ({N_SPLITS}-fold) │ Precision={avg_p:.4f}±{std_p:.4f}"
              f"  Recall={avg_r:.4f}±{std_r:.4f}  F1={avg_f1:.4f}±{std_f1:.4f}")

        # Normalised confusion matrix (sum across folds, then normalise)
        cm_norm = cm_sum / cm_sum.sum(axis=1, keepdims=True)
        print(f"\n  Normalised Confusion Matrix (summed over {N_SPLITS} folds):")
        print(f"  {'':15}  Pred:Normal  Pred:Anomaly")
        print(f"  {'Actual:Normal':15}  {cm_norm[0,0]:.4f}       {cm_norm[0,1]:.4f}")
        print(f"  {'Actual:Anomaly':15}  {cm_norm[1,0]:.4f}       {cm_norm[1,1]:.4f}")

        results[clf_name] = {
            "fold_cms":  fold_cms,
            "cm_sum":    cm_sum,
            "avg_p":     avg_p, "std_p": std_p,
            "avg_r":     avg_r, "std_r": std_r,
            "avg_f1":    avg_f1,"std_f1": std_f1,
        }

    return results


# ═══════════════════════════════════════════════════════════════════════════════
# 5. VISUALISATION
# ═══════════════════════════════════════════════════════════════════════════════

def plot_results(results, out_dir="."):
    clf_names = list(results.keys())
    n = len(clf_names)

    # ── (a) Normalised confusion matrices ────────────────────────────────────
    fig, axes = plt.subplots(1, n, figsize=(6*n, 5))
    if n == 1:
        axes = [axes]
    for i, name in enumerate(clf_names):
        plot_normalized_cm(results[name]["cm_sum"], name, axes, i)
    fig.suptitle(f"HDFS v1 – Normalised Confusion Matrices\n"
                 f"({N_SPLITS}-fold Stratified K-Fold, averaged)", fontsize=12)
    plt.tight_layout()
    cm_path = os.path.join(out_dir, "confusion_matrices.png")
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n[INFO] Confusion matrix plot saved → {cm_path}")

    # ── (b) Summary bar chart: Precision / Recall / F1 ───────────────────────
    metrics = ["avg_p", "avg_r", "avg_f1"]
    labels  = ["Precision", "Recall", "F1"]
    errs    = ["std_p",  "std_r",  "std_f1"]
    x       = np.arange(len(clf_names))
    width   = 0.25
    colors  = ["#4C72B0", "#DD8452", "#55A868"]

    fig, ax = plt.subplots(figsize=(10, 5))
    for j, (m, lbl, e, col) in enumerate(zip(metrics, labels, errs, colors)):
        vals = [results[n][m] for n in clf_names]
        stds = [results[n][e] for n in clf_names]
        bars = ax.bar(x + j*width, vals, width, label=lbl,
                      color=col, alpha=0.85, yerr=stds, capsize=4)
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                    f"{h:.3f}", ha="center", va="bottom", fontsize=7)

    ax.set_xticks(x + width)
    ax.set_xticklabels([n.replace(" ", "\n") for n in clf_names], fontsize=9)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Score")
    ax.set_title(f"HDFS v1 – Average Precision / Recall / F1  "
                 f"({N_SPLITS}-fold Stratified CV)")
    ax.legend(loc="lower right")
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    bar_path = os.path.join(out_dir, "metrics_summary.png")
    plt.savefig(bar_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[INFO] Metrics summary plot saved → {bar_path}")

    # ── (c) Per-fold F1 line plot ─────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 4))
    fold_labels = [f"Fold {i+1}" for i in range(N_SPLITS)]
    markers = ["o", "s", "D"]
    for idx, (name, res) in enumerate(results.items()):
        fold_f1s = [f1_score(
                        np.zeros(1),  # placeholder; we reuse stored fold cms
                        np.zeros(1)
                    ) for _ in range(N_SPLITS)]
        # reconstruct fold-level F1 from stored CMs
        for fi, cm in enumerate(res["fold_cms"]):
            tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0,0,0,0)
            denom = 2*tp + fp + fn
            fold_f1s[fi] = (2*tp / denom) if denom > 0 else 0.0
        ax.plot(fold_labels, fold_f1s, marker=markers[idx],
                label=name, linewidth=1.8)

    ax.set_ylim(0, 1.05)
    ax.set_ylabel("F1 Score")
    ax.set_title("Per-Fold F1 Score – All Classifiers")
    ax.legend(fontsize=8)
    ax.grid(linestyle="--", alpha=0.5)
    plt.tight_layout()
    fold_path = os.path.join(out_dir, "per_fold_f1.png")
    plt.savefig(fold_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[INFO] Per-fold F1 plot saved       → {fold_path}")


# ═══════════════════════════════════════════════════════════════════════════════
# 6. FINAL SUMMARY TABLE
# ═══════════════════════════════════════════════════════════════════════════════

def print_summary_table(results):
    print(f"\n{'═'*72}")
    print("  FINAL SUMMARY TABLE  (mean ± std over {}-folds)".format(N_SPLITS))
    print(DIVIDER)
    print(f"  {'Classifier':<35} {'Precision':>12} {'Recall':>12} {'F1':>12}")
    print(DIVIDER)
    for name, res in results.items():
        p  = f"{res['avg_p']:.4f}±{res['std_p']:.4f}"
        r  = f"{res['avg_r']:.4f}±{res['std_r']:.4f}"
        f1 = f"{res['avg_f1']:.4f}±{res['std_f1']:.4f}"
        print(f"  {name:<35} {p:>12} {r:>12} {f1:>12}")
    print(DIVIDER)


# ═══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("=" * 72)
    print("  HDFS v1 Anomaly Detection – Multi-Classifier Evaluation")
    print(f"  Strategy : 1-cross × {N_SPLITS}-fold Stratified K-Fold")
    print(f"  Data dir : {os.path.abspath(DATA_DIR)}")
    print("=" * 72)

    X, y, feature_names = load_dataset(DATA_DIR)

    classifiers = build_classifiers()

    print(f"\n[INFO] Running {N_SPLITS}-fold Stratified K-Fold …")
    results = run_pipeline(X, y, classifiers)

    print_summary_table(results)

    out_dir = "."
    plot_results(results, out_dir=out_dir)

    print("\n[DONE] All results printed above. Plots saved to current directory.\n")

  HDFS v1 Anomaly Detection – Multi-Classifier Evaluation
  Strategy : 1-cross × 5-fold Stratified K-Fold
  Data dir : c:\Users\nikhi\OneDrive\Attachments\Desktop\Git_Repos\Log_Anomly_Detection\Dataset\HDFS_v1\preprocessed
[INFO] Loading from .\HDFS.npz …
       NPZ keys: ['x_data', 'y_data']
[WARN] NPZ found but expected keys missing; falling back to Excel.
[INFO] Loading from CSV files …
[INFO] Missing values detected. Filling NaNs with 0.
       X shape: (575061, 30), y shape: (575061,)
       Class distribution – Normal: 558223, Anomaly: 16838

[INFO] Running 5-fold Stratified K-Fold …

════════════════════════════════════════════════════════════════════════
  CLASSIFIER: SVM (RBF)
────────────────────────────────────────────────────────────────────────
  Fold  1 │ Precision=0.9909  Recall=0.9997  F1=0.9953
  Fold  2 │ Precision=0.9920  Recall=1.0000  F1=0.9960
  Fold  3 │ Precision=0.9929  Recall=1.0000  F1=0.9964
  Fold  4 │ Precision=0.9950  Recall=1.0000  F1=0.9975
  Fold  5 │ 

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

# 1. Load and Prepare Data
print("Loading HDFS CSV files...")
X_df = pd.read_csv('Event_occurrence_matrix.csv')
y_df = pd.read_csv('anomaly_label.csv')

# Remove non-numeric ID columns
if 'BlockId' in X_df.columns:
    X_df = X_df.drop(columns=['BlockId'])

# Robust feature selection (Numeric only)
X_numeric = X_df.select_dtypes(include=[np.number])
X = X_numeric.values

# Encode labels
le = LabelEncoder()
label_col = 'Label' if 'Label' in y_df.columns else y_df.columns[1]
y = le.fit_transform(y_df[label_col])

# 2. 5-Fold Stratified Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_clf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)

all_cms = []

print("\n--- Training Progress ---")
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    rf_clf.fit(X_train, y_train)
    y_pred = rf_clf.predict(X_test)
    
    # Print individual fold metrics
    print(f"\n[FOLD {fold}]")
    print(classification_report(y_test, y_pred, target_names=le.classes_.astype(str)))
    
    all_cms.append(confusion_matrix(y_test, y_pred))

# 3. Final Aggregation
total_cm = np.sum(all_cms, axis=0)

# --- 4. Plot & Save RAW Matrix (randomforest.png) ---
plt.figure(figsize=(8, 6))
sns.heatmap(total_cm, annot=True, fmt='d', cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('HDFS RF: Raw Count Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.savefig('randomforest.png', dpi=300, bbox_inches='tight')
plt.show()

# --- 5. FIXED NORMALIZATION (randomforest(n).png) ---
# Sum across rows to get total actual counts per class
row_sums = total_cm.sum(axis=1)[:, np.newaxis]

# Use np.divide to avoid errors if a row sum is 0 (though rare in HDFS)
cm_norm = np.divide(total_cm.astype('float'), row_sums, 
                    out=np.zeros_like(total_cm, dtype=float), 
                    where=row_sums != 0)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Greens",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('HDFS RF: Normalized Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.savefig('randomforest(n).png', dpi=300, bbox_inches='tight')
plt.show()

print("\nProcessing Complete.")
print("Files saved: 'randomforest.png' and 'randomforest(n).png'")

Loading HDFS CSV files...

--- Training Progress ---

[FOLD 1]
              precision    recall  f1-score   support

     Anomaly       1.00      1.00      1.00      3368
      Normal       1.00      1.00      1.00    111645

    accuracy                           1.00    115013
   macro avg       1.00      1.00      1.00    115013
weighted avg       1.00      1.00      1.00    115013


[FOLD 2]
              precision    recall  f1-score   support

     Anomaly       1.00      1.00      1.00      3367
      Normal       1.00      1.00      1.00    111645

    accuracy                           1.00    115012
   macro avg       1.00      1.00      1.00    115012
weighted avg       1.00      1.00      1.00    115012


[FOLD 3]
              precision    recall  f1-score   support

     Anomaly       1.00      1.00      1.00      3367
      Normal       1.00      1.00      1.00    111645

    accuracy                           1.00    115012
   macro avg       1.00      1.00      1.00  